# Bank Customer Churn — Prediction Model
Notebook ini melatih dan mengevaluasi model prediksi churn nasabah bank menggunakan Logistic Regression dan Random Forest.

Dataset: `Bank_Customer_Churn_Prediction.csv` (10.000 baris, churn rate 20.37%)

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

df = pd.read_csv('../data/Bank_Customer_Churn_Prediction.csv')
df.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 1. Encoding & Persiapan Data

In [2]:
df = df.drop('customer_id', axis=1)  # buang ID, tidak prediktif

le_country = LabelEncoder()
le_gender = LabelEncoder()
df['country'] = le_country.fit_transform(df['country'])
df['gender'] = le_gender.fit_transform(df['gender'])
df.head()

,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,619,0,0,42,2,0.00,1,1,1,101348.88,1
1,608,2,0,41,1,83807.86,1,0,1,112542.58,0
2,502,0,0,42,8,159660.80,3,1,0,113931.57,1
3,699,0,0,39,1,0.00,2,0,0,93826.63,0
4,850,2,0,43,2,125510.82,1,1,1,79084.10,0


## 2. Split Data
`stratify=y` penting di sini karena churn rate cuma 20.37% (data imbalanced), jadi proporsi churn di train/test harus tetap seimbang.

In [3]:
X = df.drop('churn', axis=1)
y = df['churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Churn rate train: {y_train.mean():.2%}, test: {y_test.mean():.2%}")

Train: (8000, 10), Test: (2000, 10)
Churn rate train: 20.38%, test: 20.35%


## 3. Latih Model

In [4]:
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train, y_train)
pred_lr = model_lr.predict(X_test)

model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)
pred_rf = model_rf.predict(X_test)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 4. Evaluasi
Karena churn rate cuma ~20%, fokus ke **Recall** kelas churn (label 1) — model yang gagal mendeteksi nasabah yang akan churn jauh lebih merugikan bisnis daripada salah tebak nasabah loyal.

In [5]:
print("=== Logistic Regression ===")
print(classification_report(y_test, pred_lr, digits=3))
print("=== Random Forest ===")
print(classification_report(y_test, pred_rf, digits=3))

=== Logistic Regression ===
              precision    recall  f1-score   support

           0      0.817     0.975     0.889      1593
           1      0.596     0.145     0.233       407

    accuracy                          0.806      2000
   macro avg      0.706     0.560     0.561      2000
weighted avg      0.772     0.806     0.756      2000

=== Random Forest ===
              precision    recall  f1-score   support

           0      0.875     0.967     0.919      1593
           1      0.782     0.459     0.579       407

    accuracy                          0.864      2000
   macro avg      0.829     0.713     0.749      2000
weighted avg      0.856     0.864     0.850      2000



**Hasil:**
| Model | Accuracy | Precision (churn) | Recall (churn) | F1 (churn) |
|---|---|---|---|---|
| Logistic Regression | 80.6% | 0.596 | 0.145 | 0.233 |
| Random Forest | 86.4% | 0.782 | 0.459 | 0.579 |

Random Forest jauh lebih baik dalam mendeteksi nasabah yang benar-benar akan churn (Recall 45.9% vs 14.5%), meski masih ada ruang perbaikan (misalnya lewat tuning threshold atau teknik oversampling seperti SMOTE untuk data imbalanced).

## 5. Feature Importance

In [6]:
importances = pd.Series(model_rf.feature_importances_, index=X.columns)
importances.sort_values(ascending=False)

age                 0.239934
estimated_salary    0.147069
credit_score        0.144104
balance             0.141194
products_number     0.129134
tenure              0.081958
active_member       0.039596
country             0.038467
credit_card         0.019583
gender              0.018959
dtype: float64

**3 fitur teratas:** `age` (24.0%), `estimated_salary` (14.7%), `credit_score` (14.4%) — diikuti `balance` (14.1%) dan `products_number` (12.9%).

Ini sejalan dengan insight bisnis di Bagian 5: usia (kategori Senior) dan jumlah produk terbukti berkorelasi kuat dengan churn.